# SAM 2 for Roman Space Telescope: Tutorial 02
## Training and Fine-Tuning SAM 2 with a Single Astronomical Dataset

This tutorial explains **how to fine-tune Meta's Segment Anything Model 2 (SAM 2)** on astronomical data, detailing:
1. **What things you need** to fine-tune SAM 2.
2. **Expected data formats** (master FITS image tiling vs directory structure, integer labeled segmaps).
3. **Organization of the training pipeline** (freeze strategy, prompt generation, loss function, validation).
4. **Parameters of the training** (learning rate, weight decay, batch size, prompt types).
5. **Complete, runnable training walkthrough** with checkpoint saving and post-training inference.

---
## Part 1: What Things Do You Need?

To train/fine-tune SAM 2 on astronomical imaging, you need 5 core elements:

1. **Astronomical Image Data**:
   - FITS files containing single-band float32 count rates or electron counts (e.g. Roman WFI filters like F106, F129, F158, or HST/JWST imaging).
   - Can be **one large master detector image** (e.g., $4088 \times 4088$ pixels) or a collection of cutouts (e.g., $512 \times 512$ pixels).

2. **Ground-Truth Segmentation Annotations**:
   - An integer-labeled 2D array or FITS `SEGMAP` HDU matching the image dimensions:
     - `0` = Sky background
     - `1, 2, ..., N` = Discrete astronomical sources (stars and galaxies).
   - Can come from Source Extractor (`SExtractor`), Astropy `photutils` (e.g. `detect_sources`), simulated Roman catalogs, or manual labeling.

3. **Prompts (Automatic in `sam2roman`)**:
   - SAM 2 is fundamentally a prompt-conditioned foundation model. During training, each forward pass needs a pair: `(Prompt, Ground-Truth Mask)`.
   - `AstronomicalDataset` automatically derives prompt points or bounding boxes directly from the ground-truth masks!

4. **Pretrained SAM 2 Foundation Weights**:
   - Downloaded automatically from Hugging Face Hub (`facebook/sam2.1-hiera-tiny`, `facebook/sam2.1-hiera-small`, etc.).

5. **Compute Hardware**:
   - Apple Silicon GPU (Metal Performance Shaders / MPS) or an NVIDIA CUDA GPU.

---
## Part 2: Expected Formats & Organization

### Format Option A: Single Master Exposure with Tiling (Recommended for 1 Dataset)
When you have **one** deep pointing (e.g., a $4088 \times 4088$ Roman WFI exposure and its catalog/segmap), `sam2roman` slices it into overlapping tiles (e.g. $512 \times 512$ with stride 256 or 384).
A single $4088 \times 4088$ image immediately yields **$50 - 150$ training cutouts**!

### Format Option B: Multi-File Directory
```text
my_dataset/
├── images/
│   ├── patch_001.fits
│   ├── patch_002.fits
└── masks/
    ├── patch_001_segmap.npy  (or .fits)
    ├── patch_002_segmap.npy
```

### Training Organization Architecture
```text
Master FITS + SegMap  -->  AstronomicalDataset.from_single_fits_tiled()
                                      │
                    ┌─────────────────┴─────────────────┐
                    ▼                                   ▼
               Train Dataset (80%)                Val Dataset (20%)
                    │                                   │
                    ▼                                   ▼
              Train DataLoader                    Val DataLoader
                    │                                   │
                    ▼                                   ▼
            SAM2Trainer Engine ────────────────> Validation Metrics
            ├─ Vision Backbone: FROZEN          ├─ Val Loss (BCE + Dice)
            ├─ Prompt Encoder:  TRAINABLE       └─ Val Mean IoU
            ├─ Mask Decoder:    TRAINABLE               │
            └─ Loss: Combined BCE + Dice                ▼
                                                 Save Checkpoints
                                              (checkpoint_best.pt)
```

---
## Part 3: Parameters of Training

| Parameter | Recommended Value | Purpose & Rationale |
| :--- | :---: | :--- |
| `model_id` | `"facebook/sam2.1-hiera-tiny"` | Lightweight checkpoint for fast training; scale to `small` or `large` for production. |
| `freeze_vision_encoder` | `True` | **Crucial**: Freezes the ~90% of model weights in Hiera ViT. Preserves general features, prevents overfitting on small datasets, and uses $<2$ GB memory. |
| `learning_rate` | `1e-4` | Standard AdamW learning rate for the prompt encoder and mask decoder heads. |
| `weight_decay` | `1e-4` | L2 weight regularization to prevent memorization of noise. |
| `batch_size` | `2` to `4` | Cutouts are $512 \times 512$. Batch size of 2–4 fits comfortably on Apple Silicon MPS or 8GB GPUs. |
| `epochs` | `5` to `15` | Because the vision encoder is frozen, the lightweight decoder converges rapidly in ~5–10 epochs. |
| `prompt_type` | `'point'` or `'box'` | Type of prompt generated during training. `'point'` trains the model to segment from single clicks; `'box'` trains it to segment from detection boxes. |
| `box_jitter_pct` | `0.05` ($5\%$) | Slight coordinate perturbation added to bounding boxes so the model learns tolerance to imperfect catalog detections. |
| `stretch_type` | `"asinh"` | Lupton asinh stretch ($a=0.05$) to compress high-dynamic-range fluxes into RGB. |

### Google Colab Setup (Optional)
If running in Google Colab (with A100 GPU):
```python
# !git clone https://github.com/your-username/sam2roman.git
# %cd sam2roman
# !pip install -e .
```

### Step 1: Import Libraries and Verify Environment

In [ ]:
import sys
from pathlib import Path

for p in ['/content/sam2roman/src', 'src', '../src', '../../src']:
    resolved = Path(p).resolve()
    if resolved.exists() and str(resolved) not in sys.path:
        sys.path.insert(0, str(resolved))
        break

import matplotlib.pyplot as plt
import numpy as np
import torch

import sam2roman
from sam2roman.io.fits import FITSImage
from sam2roman.training.dataset import AstronomicalDataset
from sam2roman.training.trainer import SAM2Trainer
from sam2roman.inference.segmentor import SAM2Segmentor
from sam2roman.utils.synthetic import generate_synthetic_roman_fits

print(f"sam2roman version: {sam2roman.__version__}")
print(f"PyTorch version:   {torch.__version__}")
print(f"Device:            {'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'}")

### Step 2: Prepare the Single Master Dataset
Here we create a synthetic Roman WFI deep field ($1024 \times 1024$) with ground-truth stars, Sérsic galaxies, and an integer-labeled `SEGMAP`.

In [ ]:
fits_path = Path("sample_data/sample_roman_train_field.fits")

if not fits_path.exists():
    image_data, header, catalog, gt_segmap = generate_synthetic_roman_fits(
        output_path=fits_path,
        shape=(1024, 1024),
        n_stars=60,
        n_galaxies=35,
        seed=42,
        filter_name="F158",
    )
else:
    fits_obj = FITSImage.from_file(fits_path)
    image_data = fits_obj.data
    _, _, catalog, gt_segmap = generate_synthetic_roman_fits(
        shape=(1024, 1024), n_stars=60, n_galaxies=35, seed=42
    )

print(f"Master image shape: {image_data.shape}")
print(f"Ground truth sources in field: {len(catalog)}")

### Step 3: Create Tiled Cutouts from the Master Image
We use `AstronomicalDataset.from_single_fits_tiled` to slice our single field into overlapping $512 \times 512$ patches with stride $256$, filtering out empty background tiles.

In [ ]:
full_dataset = AstronomicalDataset.from_single_fits_tiled(
    fits_image=image_data,
    segmap=gt_segmap,
    tile_size=512,
    stride=256,
    min_objects=2,
    prompt_type="point",  # Generates point prompts from foreground masks
    stretch_type="asinh",
)

print(f"Created {len(full_dataset)} training tiles from 1 single master image!")

# Split into 80% Train, 20% Validation
n_train = max(1, int(0.8 * len(full_dataset)))
n_val = len(full_dataset) - n_train
train_ds, val_ds = torch.utils.data.random_split(
    full_dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

### Step 4: Inspect a Training Cutout Sample
Let's visualize one cutout: the asinh-stretched image, the ground-truth target mask, and the automatically sampled prompt point.

In [ ]:
sample = train_ds[0]
img_pil = sample["pil_image"]
mask_np = sample["mask"][0].numpy()
pt = sample["point"][0].tolist()
box = sample["box"].tolist()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(img_pil)
axes[0].plot(pt[0], pt[1], 'r*', markersize=14, label="Point Prompt")
rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], fill=False, edgecolor='yellow', linewidth=1.5, linestyle='--', label="Box Prompt")
axes[0].add_patch(rect)
axes[0].set_title("Training Tile with Derived Prompts")
axes[0].legend(loc="upper right")

axes[1].imshow(mask_np, cmap="gray")
axes[1].plot(pt[0], pt[1], 'r*', markersize=14)
axes[1].set_title("Ground Truth Target Mask")

overlay = np.array(img_pil).astype(float) / 255.0
overlay[mask_np > 0, 0] = 1.0  # Highlight object in red
axes[2].imshow(overlay)
axes[2].set_title("Overlay Verification")

plt.tight_layout()
plt.show()

### Step 5: Configure `SAM2Trainer`
We freeze the vision encoder and set up AdamW optimization with Combined BCE + Dice Loss.

In [ ]:
trainer = SAM2Trainer(
    model_id="facebook/sam2.1-hiera-tiny",
    learning_rate=1e-4,
    weight_decay=1e-4,
    freeze_vision_encoder=True,
    prompt_type="point",
)

print(f"Device:               {trainer.device}")
print(f"Trainable parameters: {sum(p.numel() for p in trainer.trainable_params):,}")

### Step 6: Execute the Fine-Tuning Loop
We run fine-tuning for 4 epochs, tracking both training loss, validation loss, and validation mean IoU.

In [ ]:
output_dir = Path("checkpoints")
output_dir.mkdir(parents=True, exist_ok=True)

history = trainer.train(
    dataset=train_ds,
    val_dataset=val_ds,
    epochs=4,
    batch_size=2,
    output_dir=output_dir,
    verbose=True,
)

### Step 7: Plot Training and Validation Convergence

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs, history["train_loss"], 'o-', label="Train Loss (BCE + Dice)", color="royalblue")
ax1.plot(epochs, history["val_loss"], 's--', label="Validation Loss", color="crimson")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Convergence Curves")
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(epochs, history["val_iou"], '^-', label="Validation Mean IoU", color="forestgreen")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Mean IoU")
ax2.set_title("Segmentation Quality (IoU)")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

### Step 8: Test Inference with the Fine-Tuned Checkpoint

In [ ]:
best_checkpoint = output_dir / "checkpoint_best.pt"
if not best_checkpoint.exists():
    best_checkpoint = output_dir / "checkpoint_epoch_4.pt"

# Load fine-tuned weights into SAM2Segmentor
segmentor = SAM2Segmentor(model_id="facebook/sam2.1-hiera-tiny")
segmentor.model_wrapper.model.load_state_dict(
    torch.load(best_checkpoint, map_location=trainer.device)["model_state_dict"]
)
segmentor.model_wrapper.model.eval()
print(f"Loaded fine-tuned model weights from: {best_checkpoint}")

# Test on a validation cutout
val_sample = val_ds[0]
test_patch = val_sample["raw_data"]
target_pt = val_sample["point"][0].tolist()

res = segmentor.segment_points(
    image=test_patch,
    points=[(target_pt[0], target_pt[1])],
    stretch_type="asinh",
    verbose=True,
)

# Visualize fine-tuned prediction
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(val_sample["pil_image"])
ax1.plot(target_pt[0], target_pt[1], 'r*', markersize=14, label="Prompt Point")
ax1.set_title("Test Image + Prompt")
ax1.legend()

ax2.imshow(val_sample["pil_image"])
if res.masks:
    pred_mask = res.masks[0]
    overlay = np.zeros((*pred_mask.shape, 4))
    overlay[pred_mask] = [0.2, 0.8, 0.2, 0.6]  # Green overlay
    ax2.imshow(overlay)
ax2.set_title(f"Fine-Tuned Mask (Area: {res.areas[0]} px, Score: {res.scores[0]:.2f})")
plt.tight_layout()
plt.show()